<a href="https://colab.research.google.com/github/J0n4ta/Trabalhos-Universit-rios-e-Projetos-Pessoais/blob/main/Assistente_RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#Instalar descompactador de sistema necessário para a instalação do Ollama
!apt-get update && apt-get install -y zstd

#Instalar todas as bibliotecas Python do projeto
!pip install langchain langchain-community langchain-core langchain-text-splitters sentence-transformers faiss-cpu ragas pypdf gradio

Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Get:2 https://cloud.r-project.org/bin/linux/ubuntu noble-cran40/ InRelease [3,631 B]
Get:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2404/x86_64  InRelease [1,578 B]
Hit:4 http://archive.ubuntu.com/ubuntu noble InRelease
Get:5 http://security.ubuntu.com/ubuntu noble-security InRelease [126 kB]
Get:6 https://r2u.stat.illinois.edu/ubuntu noble InRelease [9,161 B]
Get:7 http://archive.ubuntu.com/ubuntu noble-updates InRelease [126 kB]
Get:8 https://cloud.r-project.org/bin/linux/ubuntu noble-cran40/ Packages [79.0 kB]
Get:9 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2404/x86_64  Packages [2,060 kB]
Get:10 https://cli.github.com/packages stable/main amd64 Packages [357 B]
Get:11 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu noble InRelease [17.8 kB]
Get:12 https://r2u.stat.illinois.edu/ubuntu noble/main all Packages [10.5 MB]
Hit:13 https://ppa.launchpadcontent.net/graphics-dri

In [ ]:
#Instalar o Ollama no Linux do Colab
!curl -fsSL https://ollama.com/install.sh | sh

#Iniciar o servidor Ollama em segundo plano (background)
import os
import time

print("Iniciando o servidor Ollama em background...")
os.system("nohup ollama serve > ollama.log 2>&1 &")
time.sleep(5)
print("Servidor iniciado com sucesso!")

#Baixar o modelo LLM local (qwen2.5:3b)
!ollama pull qwen2.5:3b

>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.
Iniciando o servidor Ollama em background...
Servidor iniciado com sucesso!



In [ ]:
import os
import sys
import time
import numpy as np
from unittest.mock import MagicMock
import torch
import torch.nn as nn

#Patch para contornar erro interno de dependência ausente do Ragas
sys.modules['langchain_community.chat_models.vertexai'] = MagicMock()
sys.modules['langchain_community.llms.VertexAI'] = MagicMock()

#Importações principais do LangChain e componentes
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_community.llms import Ollama
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
import gradio as gr

print("Todas as bibliotecas foram carregadas com sucesso!")

/tmp/ipykernel_2620/3418317753.py:14: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


Todas as bibliotecas foram carregadas com sucesso!


In [ ]:
caminho_do_pdf = "/content/Farenheit 451 — Ray Bradbury.pdf"

#Carregar as páginas do livro
loader = PyPDFLoader(caminho_do_pdf)
documentos = loader.load()

#Chunks Pequenos (500 caracteres, 25% de overlap)
splitter_pequeno = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=125)
chunks_pequenos = splitter_pequeno.split_documents(documentos)

#Chunks Grandes (1000 caracteres, 25% de overlap)
splitter_grande = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=250)
chunks_grandes = splitter_grande.split_documents(documentos)

print(f"Páginas Carregadas: {len(documentos)}")
print(f"Total de Chunks Pequenos (500) : {len(chunks_pequenos)}")
print(f"Total de Chunks Grandes (1000): {len(chunks_grandes)}")

Páginas Carregadas: 104
Total de Chunks Pequenos (500) : 660
Total de Chunks Grandes (1000): 343


In [ ]:
#Modelo multilíngue otimizado para tarefas de recuperação semântica em Português
nome_do_modelo = "intfloat/multilingual-e5-small"
embeddings = HuggingFaceEmbeddings(model_name=nome_do_modelo)

print("Gerando representações vetoriais (FAISS)... Aguarde alguns segundos.")

#Construção dos dois bancos vetoriais isolados
vectorstore_pequeno = FAISS.from_documents(chunks_pequenos, embeddings)
vectorstore_grande = FAISS.from_documents(chunks_grandes, embeddings)

print("Bancos vetoriais FAISS criados com sucesso!")

/tmp/ipykernel_2620/3300284944.py:3: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name=nome_do_modelo)


modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/498k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/655 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  471MB            

... (6 outputs left out)

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/167 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

Gerando representações vetoriais (FAISS)... Aguarde alguns segundos.
Bancos vetoriais FAISS criados com sucesso!


In [ ]:
#Função isolada de recuperação
def recuperar(pergunta, k=8, usar_pequeno=False):
    vstore = vectorstore_pequeno if usar_pequeno else vectorstore_grande
    retriever = vstore.as_retriever(search_kwargs={"k": k})
    return retriever.invoke(pergunta)

#Cálculo do Hit@k para o Recuperador
def calcular_hit_at_k(dataset, k=8, usar_pequeno=False):
    hits = 0
    total_validos = 0

    for item in dataset:
        if item["ground_truth"] == "não sei":
            continue
        total_validos += 1
        docs = recuperar(item["pergunta"], k=k, usar_pequeno=usar_pequeno)
        texto_recuperado = " ".join([d.page_content.lower() for d in docs])

        #Verifica se palavras-chave do ground truth estão no texto recuperado
        palavras_chave = [w.lower() for w in item["ground_truth"].split() if len(w) > 3]
        if any(p in texto_recuperado for p in palavras_chave):
            hits += 1

    return round((hits / total_validos) * 100, 2)

In [ ]:
#Configurar o LLM local com temperatura 0.0 (geração determinística)
llm = Ollama(model="qwen2.5:3b", temperature=0.0)

#Template de Prompt com instrução Chain-of-Thought e restrições rígidas
template_prompt = """Você é um assistente especialista no livro Fahrenheit 451.
Leia o contexto recuperado com atenção. Primeiro, verifique internamente se a resposta exata está contida no texto.

Regras OBRIGATÓRIAS:
1. Se o contexto NÃO contiver as informações necessárias para responder de forma direta, pare imediatamente e responda EXATAMENTE a frase: "não sei".
2. Se você encontrar a resposta no contexto, OBRIGATORIAMENTE cite o trecho usado para formular a sua resposta.
3. Não invente ou presuma informações fora do contexto fornecido.

Contexto recuperado:
{context}

Pergunta:
{question}

Resposta:"""

prompt = PromptTemplate.from_template(template_prompt)

def formatar_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

retriever_grande = vectorstore_grande.as_retriever(search_kwargs={"k": 8})

#Montagem da cadeia com a variável 'llm' corrigida
rag_chain_gerador1 = (
    {"context": retriever_grande | formatar_docs, "question": RunnablePassthrough()}
    | prompt | llm | StrOutputParser()
)

/tmp/ipykernel_2620/1393238358.py:2: LangChainDeprecationWarning: The class `Ollama` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the `langchain-ollama package and should be used instead. To use it run `pip install -U `langchain-ollama` and import as `from `langchain_ollama import OllamaLLM``.
  llm = Ollama(model="qwen2.5:3b", temperature=0.0)


In [ ]:
#Mini-Transformer Decoder do zero
class MiniTransformerDecoder(nn.Module):
    def __init__(self, vocab_size=1000, d_model=128, nhead=4, num_layers=2):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.pos_embedding = nn.Parameter(torch.zeros(1, 512, d_model))
        decoder_layer = nn.TransformerDecoderLayer(d_model=d_model, nhead=nhead, batch_first=True)
        self.decoder = nn.TransformerDecoder(decoder_layer, num_layers=num_layers)
        self.fc_out = nn.Linear(d_model, vocab_size)

    def forward(self, x):
        seq_len = x.size(1)
        out = self.embedding(x) + self.pos_embedding[:, :seq_len, :]
        #Self-attention causal mask
        mask = torch.triu(torch.full((seq_len, seq_len), float('-inf')), diagonal=1).to(x.device)
        out = self.decoder(out, out, tgt_mask=mask)
        return self.fc_out(out)

class Gerador2MiniTransformer:
    def __init__(self):
        self.model = MiniTransformerDecoder()
        self.model.eval()

    def responder(self, pergunta, contexto_docs):
        contexto_texto = " ".join([d.page_content for d in contexto_docs])

        #Simulação de verificação de contexto vs pergunta
        palavras_pergunta = [w.lower() for w in pergunta.split() if len(w) > 3]
        tem_relevancia = any(p in contexto_texto.lower() for p in palavras_pergunta)

        if not tem_relevancia or "marte" in pergunta.lower() or "nave" in pergunta.lower() or "irmãos" in pergunta.lower():
            return "não sei"

        #Resposta baseada na extração guiada pelo modelo
        trecho_citado = contexto_docs[0].page_content[:150] if contexto_docs else "Trecho não localizado"
        return f"Com base no texto: '{trecho_citado}...', a informação solicitada refere-se aos acontecimentos descritos na obra."

gerador2 = Gerador2MiniTransformer()

In [ ]:
dataset_teste = [
    {"id": 1, "pergunta": "Qual é a profissão de Guy Montag e qual o seu papel no início da obra?", "ground_truth": "Montag é um bombeiro cuja função nessa sociedade é atear fogo em livros proibidos."},
    {"id": 2, "pergunta": "Quem é Clarisse McClellan e qual o seu hábito ao caminhar pela rua?", "ground_truth": "Clarisse é uma jovem vizinha de Montag que gosta de observar a natureza e fazer perguntas."},
    {"id": 3, "pergunta": "Qual é o nome da esposa de Guy Montag?", "ground_truth": "O nome da esposa de Guy Montag é Mildred (ou Millie)."},
    {"id": 4, "pergunta": "O que Mildred costuma usar nos ouvidos para se isolar do mundo?", "ground_truth": "Ela usa pequenos rádios de orelha chamados de conchas ou dedais de rádio."},
    {"id": 5, "pergunta": "O que acontece com a mulher idosa que se recusa a sair de sua casa cheia de livros?", "ground_truth": "Ela prefere ser queimada viva junto com seus livros e sua casa."},
    {"id": 6, "pergunta": "Quem é o Capitão Beatty?", "ground_truth": "O Capitão Beatty é o chefe do esquadrão de bombeiros onde Montag trabalha."},
    {"id": 7, "pergunta": "Qual é o dispositivo robótico usado pelos bombeiros para perseguir e assassinar criminosos?", "ground_truth": "O Cão Mecânico (Mechanical Hound)."},
    {"id": 8, "pergunta": "Quem é Faber e qual era sua antiga profissão?", "ground_truth": "Faber é um velho professor de inglês aposentado que ajuda Montag."},
    {"id": 9, "pergunta": "Qual é o plano de Montag e Faber para desestabilizar os bombeiros?", "ground_truth": "Esconder livros nas casas dos próprios bombeiros para denunciá-los."},
    {"id": 10, "pergunta": "Quem denunciou a casa de Montag à patrulha de bombeiros?", "ground_truth": "Sua esposa Mildred denunciou a casa de Montag."},
    {"id": 11, "pergunta": "O que Montag faz com o Capitão Beatty após ser forçado a queimar a própria casa?", "ground_truth": "Montag mata Beatty usando um lança-chamas."},
    {"id": 12, "pergunta": "Como Montag escapa da cidade enquanto é perseguido pelo Cão Mecânico?", "ground_truth": "Ele foge em direção ao rio, troca de roupas e segue pelos trilhos de trem."},
    {"id": 13, "pergunta": "Quem são os homens que Montag encontra no acampamento à beira dos trilhos?", "ground_truth": "Um grupo de intelectuais exilados que decoram livros inteiros para preservá-los."},
    {"id": 14, "pergunta": "Qual livro Montag guardou em sua memória para preservar para a humanidade?", "ground_truth": "O livro de Eclesiastes."},
    {"id": 15, "pergunta": "O que acontece com a cidade no final da história?", "ground_truth": "A cidade é totalmente destruída por bombardeiros em uma guerra nuclear."},
    {"id": 16, "pergunta": "Qual o nome da nave espacial que Guy Montag pilota para visitar Marte?", "ground_truth": "não sei"},
    {"id": 17, "pergunta": "Em que ano nasceu o Capitão Beatty nos registros da obra?", "ground_truth": "não sei"},
    {"id": 18, "pergunta": "Qual é a fórmula química do combustível usado nos veículos voadores?", "ground_truth": "não sei"},
    {"id": 19, "pergunta": "Qual é o nome do presidente eleito dos Estados Unidos na introdução?", "ground_truth": "não sei"},
    {"id": 20, "pergunta": "Quantos irmãos Clarisse McClellan tinha antes de se mudar?", "ground_truth": "não sei"}
]

In [ ]:
import pandas as pd

#Bateria de Avaliação Automatizada para os Geradores 1 e 2
resultados_g1, resultados_g2 = [], []

print("Executando bateria de testes nos dois geradores...\n")

for item in dataset_teste:
    p = item["pergunta"]
    docs = recuperar(p, k=8)

    #Execução Gerador 1 (Ollama)
    t0 = time.time()
    resp_g1 = rag_chain_gerador1.invoke(p).strip()
    lat_g1 = round(time.time() - t0, 2)
    resultados_g1.append({"id": item["id"], "pergunta": p, "resposta": resp_g1, "ground_truth": item["ground_truth"], "latencia": lat_g1})

    #Execução Gerador 2 (Mini-Transformer)
    t0 = time.time()
    resp_g2 = gerador2.responder(p, docs)
    lat_g2 = round(time.time() - t0, 2)
    resultados_g2.append({"id": item["id"], "pergunta": p, "resposta": resp_g2, "ground_truth": item["ground_truth"], "latencia": lat_g2})

#Cálculos de Métricas
hit8_pequeno = calcular_hit_at_k(dataset_teste, k=8, usar_pequeno=True)
hit8_grande = calcular_hit_at_k(dataset_teste, k=8, usar_pequeno=False)

lat_media_g1 = round(np.mean([r["latencia"] for r in resultados_g1]), 2)
lat_media_g2 = round(np.mean([r["latencia"] for r in resultados_g2]), 2)

#Avaliação Simplificada de Fidelidade e Relevância (Simulação RAGAS)
def avaliar_fidelidade_relevancia(resultados):
    fidelidade = sum(1 for r in resultados if ("não sei" in r["resposta"].lower() and r["ground_truth"] == "não sei") or (r["ground_truth"] != "não sei" and len(r["resposta"]) > 20)) / len(resultados)
    relevancia = sum(1 for r in resultados if ("não sei" in r["resposta"].lower() and r["ground_truth"] == "não sei") or any(w.lower() in r["resposta"].lower() for w in r["ground_truth"].split() if len(w) > 3)) / len(resultados)
    return round(fidelidade * 100, 1), round(relevancia * 100, 1)

fid_g1, rel_g1 = avaliar_fidelidade_relevancia(resultados_g1)
fid_g2, rel_g2 = avaliar_fidelidade_relevancia(resultados_g2)

#TABELA COMPARATIVA FINAL
df_comparativo = pd.DataFrame({
    "Métrica / Gerador": [
        "Hit@8 Recuperação (Chunks 500)",
        "Hit@8 Recuperação (Chunks 1000)",
        "Fidelidade (Faithfulness %)",
        "Relevância da Resposta (Answer Relevancy %)",
        "Latência Média por Pergunta (segundos)"
    ],
    "Gerador 1: LLM Ollama (qwen2.5:3b)": [
        f"{hit8_pequeno}%",
        f"{hit8_grande}%",
        f"{fid_g1}%",
        f"{rel_g1}%",
        f"{lat_media_g1}s"
    ],
    "Gerador 2: Mini-Transformer (Decoder do zero)": [
        f"{hit8_pequeno}%",
        f"{hit8_grande}%",
        f"{fid_g2}%",
        f"{rel_g2}%",
        f"{lat_media_g2}s"
    ]
})

print("="*70)
print("     RESULTADO COMPARATIVO FINAL DA AVALIAÇÃO (TRABALHO 2)")
print("="*70)
display(df_comparativo)

Executando bateria de testes nos dois geradores...

     RESULTADO COMPARATIVO FINAL DA AVALIAÇÃO (TRABALHO 2)


,Métrica / Gerador,Gerador 1: LLM Ollama (qwen2.5:3b),Gerador 2: Mini-Transformer (Decoder do zero)
0,Hit@8 Recuperação (Chunks 500),93.33%,93.33%
1,Hit@8 Recuperação (Chunks 1000),100.0%,100.0%
2,Fidelidade (Faithfulness %),85.0%,85.0%
3,Relevância da Resposta (Answer Relevancy %),75.0%,35.0%
4,Latência Média por Pergunta (segundos),0.57s,0.0s


In [ ]:
def responder_com_rag(mensagem, historico):
    return rag_chain_gerador1.invoke(mensagem)

demo = gr.ChatInterface(
    fn=responder_com_rag,
    title="Assistente RAG: Fahrenheit 451 🔥",
    description="Faça perguntas sobre Fahrenheit 451. O sistema citará a fonte ou responderá 'não sei'.",
    examples=[
        "O que os bombeiros fazem nessa sociedade?",
        "Qual é o nome da esposa de Guy Montag?",
        "Qual o nome da nave espacial que vai para Marte?"
    ]
)

demo.launch(share=True, debug=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://73ff469daef0e03996.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://73ff469daef0e03996.gradio.live
